In [2]:
import re
import os
import shutil

# AFAC 最优TSGym配置SH生成

In [87]:
test_datasets = ['NYSE']
_pred_len = 24
best_components = ['nasdaq_LTF_TSGym_False_False_RevIN_MA_False_series-encoding_MLP_null_sparse-attention_True_False_custom_ftM_sl96_ll48_pl24_dm256_el2_dl1_df1024_fc3_ebtimeF_dtTrue_Exp_epochs50_lfMAE_lr0.0001_lrsnull_0']

In [88]:
# 模板文件路径
template_path = f'/data/nishome/user1/minqi/TSGym/scripts/long_term_forecast/{test_datasets[0]}_script/TSGym_pl{_pred_len}.sh'
    
# 输出目录
output_dir = f'/data/nishome/user1/minqi/TSGym/scripts/long_term_forecast/{test_datasets[0]}_script'
script_contents = []
for i, dataset in enumerate(test_datasets):

    # 确保输出目录存在
    # if os.path.exists(output_dir):
    #     print('delete current folder!')
    #     shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    # 读取模板内容
    with open(template_path, 'r') as file:
        template_content = file.read()

    # 对于每个模型名称，生成一个 shell 脚本
    setting = '_'.join(best_components[i].split('_')[2:])
    file_name = setting
    
    model_name = '_'.join(setting.split('_')[:12])

    intros = ['x_mark', 'multi-granularity', 'Normalization', 'Decomposition', 'Channel-independent',
            'Tokenization', 'Backbone', 'Attention', 'Feature-Attention', 'Encoder-only', 'Frozen', 'Dataset',
            'Sequence Length', 'd_model', 'd_ff', 'Encoder layers',
            'Training Epochs', 'Loss Function', 'Learning Rate', 'Learning Rate Strategy']
    key = '_'.join([setting[setting.find('TSGym')+6: setting.find('ftM')-1],
                                            re.search(r'sl(\d+)', setting)[1],
                                            re.search(r'dm(\d+)', setting)[1],
                                            re.search(r'df(\d+)', setting)[1],
                                            re.search(r'el(\d+)', setting)[1],
                                            re.search(r'epochs(\d+)', setting)[1],
                                            re.search(r'lf([A-Za-z]+)', setting)[1],
                                            re.search(r'lr(\d+(?:\.\d+)?)', setting)[1],
                                            re.search(r'lrs([A-Za-z]+\d*)', setting)[1]])
    record = {}
    for n, param in enumerate(key.split('_')):
        record.update({intros[n]:param})

    # 替换模型名称
    script_content = template_content.replace('$model_name', model_name)
    script_content = script_content.replace(f'$seq_len', record['Sequence Length'])
    script_content = script_content.replace(f'$d_model', record['d_model'])
    script_content = script_content.replace(f'$d_ff', record['d_ff'])
    script_content = script_content.replace(f'$e_layers', record['Encoder layers'])
    script_content = script_content.replace(f'$train_epochs', record['Training Epochs'])
    script_content = script_content.replace(f'$loss', record['Loss Function'])
    script_content = script_content.replace(f'$learning_rate', record['Learning Rate'])
    script_content = script_content.replace(f'$lradj', record['Learning Rate Strategy'])
    script_content = script_content.replace('$dataset', dataset)
    script_content = script_content.replace('$model_id', f'{dataset}_14_7')
    script_contents.append(script_content)
    
# 定义输出文件名
output_file = os.path.join(output_dir, f'TSGym_meta_addnew_{_pred_len}.sh')

# 写入新的 shell 脚本
with open(output_file, 'w') as file:
    # 写入 shell 脚本的 shebang
    file.write("#!/bin/bash\n\n")
    
    # 写入每个 Python 命令
    for command in script_contents:
        file.write(command + "\n\n")

# print(f'Generated {output_file}')